## Preprocess the merged data

In [3]:
%pip install -q -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [4]:
# import seaborn as sns
# import matplotlib.pyplot as plt
import pandas as pd
import os

In [12]:
# Load CSV files
data_dir = '../../mcphases/'

merged = pd.read_csv(os.path.join(data_dir, 'merged/physical_activity_merged.csv'))

print("CSV file loaded successfully!")

CSV file loaded successfully!


### 1. Examine missing values

In [13]:
# Number of missing values per column
missing_count = merged.isnull().sum()

# Percentage of missing values per column
missing_percent = ((merged.isnull().sum() / len(merged)) * 100).round(2)

# Combine into a table
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percent": missing_percent
}).sort_values("Missing Percent", ascending=False)

print(missing_summary)

                                    Missing Count  Missing Percent
pdg                                          3795            67.06
exerciselevel_num                            3002            53.05
fatigue_num                                  2328            41.14
sedentary                                    1961            34.65
sexually_active_num                           372             6.57
estrogen                                      321             5.67
lh                                            320             5.65
filtered_demographic_vo2_max                  285             5.04
cardio_zone                                   209             3.69
peak_zone                                     209             3.69
below_fat_burn_zone                           209             3.69
fat_burn_zone                                 209             3.69
very                                          178             3.15
lightly                                       178             

In [15]:
merged.shape

(5659, 25)

42 participants * 2 periods * 90 days each period = 7560 rows max

In [16]:
merged.duplicated(subset=['id','day_in_study']).sum()

np.int64(0)

### 2. Process missing values

First, fill in the 'study_interval" for time-series interpolation.

In [21]:
merged.groupby(['id', pd.cut(merged['day_in_study'], bins=[0,100,800,1004])])['study_interval'].unique()


C:\Users\caowe\AppData\Local\Temp\ipykernel_53100\1256506962.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  merged.groupby(['id', pd.cut(merged['day_in_study'], bins=[0,100,800,1004])])['study_interval'].unique()


id  day_in_study
1   (0, 100]             [2022.0]
    (100, 800]                NaN
    (800, 1004]               NaN
2   (0, 100]             [2022.0]
    (100, 800]                NaN
                        ...      
49  (100, 800]                NaN
    (800, 1004]               NaN
50  (0, 100]             [2022.0]
    (100, 800]                NaN
    (800, 1004]     [nan, 2024.0]
Name: study_interval, Length: 126, dtype: object

In [22]:
def day_range(day):
    if 1 <= day <= 100:
        return 'range_1'
    elif 800 <= day <= 1010:
        return 'range_2'
    return None

merged['_day_range'] = merged['day_in_study'].apply(day_range)

merged['study_interval'] = (
    merged.groupby(['id', '_day_range'])['study_interval']
    .transform(lambda x: x.ffill().bfill())
)

merged = merged.drop(columns='_day_range')

In [27]:
merged.isnull().sum()

id                                       0
study_interval                           0
is_weekend                               0
day_in_study                             0
sedentary                             1961
lightly                                178
moderately                             178
very                                   178
calories_sum                             4
filtered_demographic_vo2_max           285
filtered_demographic_vo2_max_error     167
peak_zone                              209
cardio_zone                            209
fat_burn_zone                          209
below_fat_burn_zone                    209
phase                                    1
lh                                     320
estrogen                               321
pdg                                   3795
exerciselevel_num                     3002
fatigue_num                           2328
age_of_first_menarche                    0
age                                      0
menstrual_h

Now, we can process other features.

In [ ]:
# 1. Exclude features
merged.drop(["sedentary"], axis=1)

,id,is_weekend,day_in_study,lightly,moderately,very,calories_sum,filtered_demographic_vo2_max,filtered_demographic_vo2_max_error,peak_zone,...,phase,lh,estrogen,pdg,exerciselevel_num,fatigue_num,age_of_first_menarche,age,menstrual_health_literacy_num,sexually_active_num
0,1,True,1,64.0,0.0,0.0,1542.00,33.79370,3.00000,0.0,...,Follicular,2.9,94.2,NaN,2.0,4.0,14,25,NaN,1.0
1,1,False,2,74.0,0.0,0.0,1591.00,32.55987,1.51239,5.0,...,Follicular,1.2,226.3,NaN,2.0,4.0,14,25,NaN,1.0
2,1,False,3,134.0,18.0,7.0,1755.00,31.50628,1.02734,5.0,...,Follicular,3.5,276.8,NaN,NaN,5.0,14,25,NaN,1.0
3,1,False,4,86.0,0.0,0.0,1552.00,31.06774,0.79267,0.0,...,Fertility,1.8,322.1,NaN,2.0,4.0,14,25,NaN,1.0
4,1,False,5,10.0,0.0,0.0,1456.00,30.88130,0.65787,8.0,...,Fertility,4.6,244.9,NaN,NaN,4.0,14,25,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5654,50,False,947,214.0,21.0,60.0,2270.14,26.04303,0.36310,1285.0,...,Luteal,4.6,70.7,2.8,NaN,NaN,11,20,2.0,0.0
5655,50,False,948,198.0,0.0,0.0,2188.42,26.05882,0.36310,1241.0,...,Luteal,5.8,87.0,7.0,NaN,NaN,11,20,2.0,0.0
5656,50,False,949,68.0,15.0,39.0,2586.10,26.06572,0.36310,1410.0,...,Luteal,NaN,NaN,NaN,NaN,NaN,11,20,2.0,0.0
5657,50,False,950,8.0,0.0,0.0,2167.06,26.03662,0.36310,1409.0,...,Menstrual,NaN,NaN,NaN,NaN,NaN,11,20,2.0,0.0


In [ ]:
# 2. Deal with the flipped time in heart rate zone features: 'peak_zone', 'cardio_zone', 'fat_burn_zone', 'below_fat_burn_zone'
# Study interval 2 reverses study interval 1's column orders
cols = ['peak_zone', 'cardio_zone', 'fat_burn_zone', 'below_fat_burn_zone']
reversed_cols = cols[::-1]  # ['below_fat_burn_zone', 'fat_burn_zone', 'cardio_zone', 'peak_zone']

mask = merged['study_interval'] == 2024.0

merged.loc[mask, cols] = merged.loc[mask, reversed_cols].values

In [30]:
merged[cols]

,peak_zone,cardio_zone,fat_burn_zone,below_fat_burn_zone
0,0.0,0.0,126.0,1036.0
1,5.0,82.0,416.0,512.0
2,5.0,119.0,599.0,368.0
3,0.0,0.0,212.0,613.0
4,8.0,123.0,250.0,308.0
...,...,...,...,...
5654,0.0,0.0,39.0,1285.0
5655,0.0,9.0,65.0,1241.0
5656,0.0,0.0,6.0,1410.0
5657,0.0,0.0,24.0,1409.0


In [31]:
# 3. Impute missing values with median
for col in ["lightly", "moderately", "very", "calories_sum", "filtered_demographic_vo2_max_error",
            'peak_zone', 'cardio_zone', 'fat_burn_zone', 'below_fat_burn_zone', "menstrual_health_literacy_num"]:
    merged[col] = merged[col].fillna(merged[col].median())

In [ ]:
# 4. Interpolate missing values for specific features
# First check that there are no duplicate day_in_study values for each id and study_interval combination
merged.groupby(['id', 'study_interval'])['day_in_study'].apply(lambda x: x.duplicated().any()).any()

np.False_

In [35]:
# Now we can safely interpolate the missing values for the specified features
features = ['filtered_demographic_vo2_max', 'phase', 'lh', 'estrogen']

merged = merged.sort_values(['id', 'study_interval', 'day_in_study'])

# Use "both" to fill NaNs at the beginning and end of the series
def interpolate_group(g):
    g = g.set_index('day_in_study')
    g[features] = g[features].interpolate(method='index', limit_direction='both')
    return g.reset_index()

merged = merged.groupby(['id', 'study_interval'], group_keys=False).apply(interpolate_group)

C:\Users\caowe\AppData\Local\Temp\ipykernel_53100\503047837.py:9: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  g[features] = g[features].interpolate(method='index', limit_direction='both')
C:\Users\caowe\AppData\Local\Temp\ipykernel_53100\503047837.py:9: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  g[features] = g[features].interpolate(method='index', limit_direction='both')
C:\Users\caowe\AppData\Local\Temp\ipykernel_53100\503047837.py:9: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  g[features] = g[features].interpolate(method='index', limit_direction='both')
C:\Users\caowe\AppData\Local\Temp\ipykernel_53100\50

In [37]:
# 5. Fill in a special value for the feature 'sexually_active_num'
merged['sexually_active_num'] = merged['sexually_active_num'].fillna(-1)

In [36]:
# Number of missing values per column
missing_count = merged.isnull().sum()

# Percentage of missing values per column
missing_percent = ((merged.isnull().sum() / len(merged)) * 100).round(2)

# Combine into a table
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percent": missing_percent
}).sort_values("Missing Percent", ascending=False)

print(missing_summary)

                                    Missing Count  Missing Percent
pdg                                          3795            67.06
exerciselevel_num                            3002            53.05
fatigue_num                                  2328            41.14
sedentary                                    1961            34.65
sexually_active_num                           372             6.57
phase                                           1             0.02
day_in_study                                    0             0.00
study_interval                                  0             0.00
id                                              0             0.00
calories_sum                                    0             0.00
very                                            0             0.00
moderately                                      0             0.00
lightly                                         0             0.00
is_weekend                                      0             